# BERT Similarity Experiments

Тот же фича-инжиниринг, что и в ML_Experiments.ipynb, включая ALS.
**Cosine similarity вычислены через BERT-модели вместо TF-IDF.**

In [ ]:
import re
import pickle
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import mlflow
import optuna

from tqdm.auto import tqdm
from scipy.sparse import csr_matrix, vstack
from transformers import AutoTokenizer, AutoModel
from implicit.als import AlternatingLeastSquares
from optuna.integration.mlflow import MLflowCallback
from mlflow.utils.mlflow_tags import MLFLOW_PARENT_RUN_ID
from sklearn.metrics import ndcg_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from lightgbm import LGBMClassifier

import nltk
import pymorphy3
from nltk.corpus import stopwords
from gensim.utils import simple_preprocess

warnings.simplefilter('ignore', FutureWarning)


In [ ]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000
EXPERIMENT_NAME = "hr-ai-scout-bert"

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")
mlflow.set_registry_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")


In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available():
    DEVICE = torch.device('mps')
else:
    DEVICE = torch.device('cpu')
print(f"Device: {DEVICE}")


## 1. Load data

In [ ]:
df = pd.read_csv('/Users/user/Documents/Magistracy/year_project/hr-ai-scout/total_df.csv')
df.head()

## 2. Preprocessing

*(скопировано из ML_Experiments.ipynb без изменений)*

In [ ]:
t1 = df.shape[0]
df = df.dropna(subset=[
    "resume_education", "resume_last_experience_description",
    "resume_last_position", "resume_last_company_experience_period",
    "resume_total_experience", "resume_experience_months",
    "resume_location", "resume_specialization",
], how="all")
print('Удалено ', t1 - df.shape[0], ' строки')


In [ ]:
t1 = df.shape[0]
df = df.loc[~(df["resume_total_experience"].notna()
              & df["resume_last_experience_description"].isna()
              & df["resume_last_position"].isna())]
df = df.loc[~(df["resume_total_experience"].isna()
              & df["resume_last_experience_description"].notna()
              & df["resume_last_position"].notna())]
print('Удалено ', t1 - df.shape[0], ' строк')


In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns
df[cat_cols] = df[cat_cols].fillna('NDT')
df['resume_age'] = df['resume_age'].fillna(df['resume_age'].mean())
df['resume_experience_months'] = df['resume_experience_months'].fillna(0)
df = df.drop(['vacancy_salary_to', 'vacancy_salary_from',
              'vacancy_salary_currency', 'vacancy_salary_gross'], axis=1)


In [ ]:
df['resume_salary_split'] = df['resume_salary'].apply(lambda x: x.split())
df['salary_int'] = df['resume_salary_split'].apply(
    lambda x: int(''.join(p for p in x if re.fullmatch(r'\d+', p)))
              if any(re.fullmatch(r'\d+', p) for p in x) else np.nan
)
currency_symbols = ['₽', '$', '€', '₴', '₸', '₼', '₾', 'Br', "so'm"]
rates_rub = {'₽':1.0,'$':80.85,'€':94.14,'₴':1.94,'₸':0.150,
             '₼':47.8,'₾':33.5,'Br':28.7,"so'm":0.0068}
df['currency_symbol'] = df['resume_salary_split'].apply(
    lambda x: next((s for s in x if s in currency_symbols), np.nan))
df['salary_converted'] = (df['salary_int'] * df['currency_symbol'].map(rates_rub)).fillna(0)
df['resume_salary'] = df['salary_converted']
df = df.drop(['resume_salary_split','salary_int','currency_symbol','salary_converted'], axis=1)


In [ ]:
def experience_to_months(text):
    months = 0
    for pat in [r'(\d+)\s*год', r'(\d+)\s*лет']:
        m = re.search(pat, text)
        if m: months += int(m.group(1)) * 12
    m = re.search(r'(\d+)\s*месяц', text)
    if m: months += int(m.group(1))
    return months if months > 0 else np.nan

df['resume_last_company_experience_months'] = \
    df['resume_last_company_experience_period'].apply(experience_to_months)
df['resume_last_company_experience_months'] = \
    df['resume_last_company_experience_months'].fillna(0)


In [ ]:
df = df[~(df.resume_salary > 1e7)]
df.loc[df['resume_experience_months'] > 720, 'resume_experience_months'] = 720
df.loc[df['resume_last_company_experience_months'] > 720, 'resume_last_company_experience_months'] = 720
df = df[~(df.resume_age > 90)]
df = df[~(df.resume_experience_months < df.resume_last_company_experience_months)]
df = df[~(df.resume_age < (df.resume_experience_months // 12) + 16)]

gender_map = {'Мужчина':'Мужчина','Male':'Мужчина','Женщина':'Женщина','Female':'Женщина'}
df['resume_gender'] = df['resume_gender'].apply(
    lambda x: gender_map.get(x, 'Неизвестно'))

print(f"Датасет после очистки: {df.shape}")


## 3. Feature engineering (без TF-IDF)

In [ ]:
# Признак совпадения локации
df['location_matching'] = (df['vacancy_area'] == df['resume_location']).astype(int)

# Количество навыков резюме в тексте вакансии
def resume_skill_count_in_vacancy(row):
    skills = row['resume_skills'].replace('[','').replace(']','').replace("'","").split(', ')
    return sum(1 for s in skills if s in row['vacancy_description'])

df['resume_skill_count_in_vacancy'] = df.apply(resume_skill_count_in_vacancy, axis=1)

# Доля слов последней должности, встречающихся в описании вакансии
def last_position_in_vacancy(row):
    bow = []
    for sep in [' ', '-', '_']:
        bow += row['resume_last_position'].split(sep=sep)
    bow = list(set(bow))
    return sum(1 for w in bow if w in row['vacancy_description']) / len(bow)

df['last_position_in_vacancy'] = df.apply(last_position_in_vacancy, axis=1)

print("Признаки добавлены: location_matching, resume_skill_count_in_vacancy, last_position_in_vacancy")


## 4. BERT Similarity

Вместо TF-IDF считаем эмбеддинги через `AutoTokenizer + AutoModel`.
Ключ оптимизации — кодируем только **уникальные** вакансии и резюме, затем маппим на все строки df.

In [ ]:
def encode_texts(texts, tokenizer, model, batch_size=64, prefix=''):
    """
    Батчевое кодирование текстов в L2-нормированные эмбеддинги.
    Mean pooling по токенам, взвешенный attention mask.
    """
    if prefix:
        texts = [prefix + t for t in texts]

    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="    encoding"):
        batch = texts[i : i + batch_size]
        encoded = tokenizer(
            batch, padding=True, truncation=True,
            max_length=512, return_tensors='pt'
        ).to(DEVICE)

        with torch.no_grad():
            out = model(**encoded)

        # Mean pooling
        token_emb = out.last_hidden_state                              # (B, T, H)
        mask = encoded['attention_mask'].unsqueeze(-1).float()         # (B, T, 1)
        pooled = (token_emb * mask).sum(1) / mask.sum(1).clamp(min=1e-9)

        pooled = F.normalize(pooled, p=2, dim=1)
        all_embeddings.append(pooled.cpu().numpy())

    return np.vstack(all_embeddings)


In [ ]:
def compute_bert_similarity(df, tokenizer, model, batch_size=64,
                             vacancy_prefix='', resume_prefix=''):
    """
    Cosine similarity между vacancy_description и resume_last_experience_description.
    Эмбеддинги вычисляются только для уникальных текстов.
    """
    df = df.copy()
    df['vacancy_description'] = df['vacancy_description'].fillna('').astype(str)
    df['resume_last_experience_description'] = \
        df['resume_last_experience_description'].fillna('').astype(str)

    # ── Уникальные вакансии ──────────────────────────────────────────
    unique_vac = df[['vacancy_id','vacancy_description']].drop_duplicates('vacancy_id')
    print(f"  Уникальных вакансий: {len(unique_vac)} / всего строк: {len(df)}")
    print("  Эмбеддинги вакансий...")
    vac_emb = encode_texts(unique_vac['vacancy_description'].tolist(),
                           tokenizer, model, batch_size, prefix=vacancy_prefix)
    vac_map = dict(zip(unique_vac['vacancy_id'], vac_emb))

    # ── Уникальные резюме ────────────────────────────────────────────
    unique_res = df[['resume_id','resume_last_experience_description']].drop_duplicates('resume_id')
    print(f"  Уникальных резюме: {len(unique_res)}")
    print("  Эмбеддинги резюме...")
    res_emb = encode_texts(unique_res['resume_last_experience_description'].tolist(),
                           tokenizer, model, batch_size, prefix=resume_prefix)
    res_map = dict(zip(unique_res['resume_id'], res_emb))

    # ── Попарное сходство (L2-норм → cosine = dot) ───────────────────
    sims = []
    for _, row in df.iterrows():
        v = vac_map.get(row['vacancy_id'])
        r = res_map.get(row['resume_id'])
        sims.append(float(np.dot(v, r)) if v is not None and r is not None else 0.0)

    return sims


In [ ]:
# (model_name, vacancy_prefix, resume_prefix, batch_size)
BERT_MODELS = [
    ('cointegrated/LaBSE-en-ru',                                    '', '',           64),
    ('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', '', '',           64),
    ('ai-forever/sbert_large_nlu_ru',                               '', '',           32),
    ('intfloat/multilingual-e5-base',                   'query: ', 'passage: ',      32),
]


In [ ]:
bert_sim_cols = {}   # model_name -> column name in df

for model_name, vac_prefix, res_prefix, bs in BERT_MODELS:
    short  = model_name.split('/')[-1].replace('-','_').lower()
    col    = f'sim_{short}'
    print(f"\n{'='*60}\n{model_name}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    bert_model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

    df[col] = compute_bert_similarity(df, tokenizer, bert_model, bs,
                                      vacancy_prefix=vac_prefix,
                                      resume_prefix=res_prefix)
    bert_sim_cols[model_name] = col
    print(f"  Готово. Среднее сходство: {df[col].mean():.4f}")

    # Освобождаем память
    del bert_model
    if DEVICE.type == 'mps':   torch.mps.empty_cache()
    elif DEVICE.type == 'cuda': torch.cuda.empty_cache()

print("\nВсе BERT-сходства вычислены:", list(bert_sim_cols.values()))


## 5. ALS

*(скопировано из ML_Experiments.ipynb без изменений)*

In [ ]:
def create_interaction_matrix(df):
    unique_vacancies = df['vacancy_id'].unique().tolist()
    unique_resumes   = df['resume_id'].unique().tolist()
    id2vacancy = dict(enumerate(unique_vacancies))
    id2resume  = dict(enumerate(unique_resumes))
    vacancy2id = {v: k for k, v in id2vacancy.items()}
    resume2id  = {v: k for k, v in id2resume.items()}
    rows = [vacancy2id[v] for v in df['vacancy_id']]
    cols = [resume2id[r]  for r in df['resume_id']]
    matrix = csr_matrix((df['target'], (rows, cols)),
                        shape=(len(unique_vacancies), len(unique_resumes)),
                        dtype=np.float32)
    return matrix, id2vacancy, id2resume, vacancy2id, resume2id, unique_vacancies, unique_resumes

def get_factors(interactions_matrix):
    als = AlternatingLeastSquares(
        factors=64, regularization=0.1, iterations=30,
        random_state=RANDOM_STATE, num_threads=0)
    als.fit(interactions_matrix.T)
    return als.item_factors, als.user_factors

def get_als_score(vacancy_id, resume_id):
    if vacancy_id not in vacancy2id or resume_id not in resume2id:
        return 0
    return float(np.dot(vacancy_factors[vacancy2id[vacancy_id]],
                         resume_factors[resume2id[resume_id]]))


## 6. Train / Test split + ALS score

In [ ]:
# Базовые признаки (без similarity — добавим bert-вариант в цикле)
BASE_FEATURES = [
    'vacancy_area', 'vacancy_experience', 'vacancy_employment', 'vacancy_schedule',
    'resume_salary', 'resume_age', 'resume_experience_months', 'resume_location',
    'resume_gender', 'resume_applicant_status', 'resume_last_company_experience_months',
    'location_matching', 'resume_skill_count_in_vacancy', 'last_position_in_vacancy',
]

categorical_features = df[BASE_FEATURES].select_dtypes(exclude=np.number).columns.tolist()
numeric_features     = df[BASE_FEATURES].select_dtypes(include=np.number).columns.tolist()

X_base = df[BASE_FEATURES].copy()
y      = df['target'].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_base, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")


In [ ]:
# ALS: обучаем на части train, чтобы избежать data leakage
X_train_als, _, y_train_als, _ = train_test_split(
    X_train, y_train, test_size=0.3, random_state=RANDOM_STATE, stratify=y_train)

train_als_interactions = df.loc[X_train_als.index, ['vacancy_id','resume_id','target']]
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, \
    unique_vacancies, unique_resumes = create_interaction_matrix(train_als_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

X_train['als_score'] = df.loc[X_train.index].apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), axis=1)

# Для test — ALS на полном train
train_interactions = df.loc[X_train.index, ['vacancy_id','resume_id','target']]
interactions_matrix, id2vacancy, id2resume, vacancy2id, resume2id, \
    unique_vacancies, unique_resumes = create_interaction_matrix(train_interactions)
vacancy_factors, resume_factors = get_factors(interactions_matrix)

X_test['als_score'] = df.loc[X_test.index].apply(
    lambda row: get_als_score(row['vacancy_id'], row['resume_id']), axis=1)

print(f"als_score в train (нули): {(X_train['als_score']==0).sum()}")
print(f"als_score в test  (нули): {(X_test['als_score']==0).sum()}")


## 7. Metrics

In [ ]:
def calculate_metrics(df_test):
    ndcg_s, prec_s, rec_s, f1_s = [], [], [], []
    for vid in df_test['vacancy_id'].unique():
        mask   = df_test['vacancy_id'] == vid
        y_true = df_test.loc[mask, 'target'].values
        y_prob = df_test.loc[mask, 'y_pred_proba'].values
        if len(y_true) <= 1:
            continue
        ndcg_s.append(ndcg_score(y_true.reshape(1,-1), y_prob.reshape(1,-1)))
        y_bin = (y_prob >= 0.5).astype(int)
        prec_s.append(precision_score(y_true, y_bin, zero_division=0))
        rec_s.append(recall_score(y_true, y_bin, zero_division=0))
        f1_s.append(f1_score(y_true, y_bin, zero_division=0))

    metrics = {
        'NDCG':      np.mean(ndcg_s),
        'Precision': np.mean(prec_s),
        'Recall':    np.mean(rec_s),
        'F1':        np.mean(f1_s),
    }
    for k, v in metrics.items():
        print(f"  {k}: {v:.4f}")
    return metrics


## 8. LightGBM + ALS + BERT Similarity

Для каждой BERT-модели запускаем Optuna (15 trials) и логируем NDCG в MLflow.

In [ ]:
try:
    experiment_id = mlflow.get_experiment_by_name(EXPERIMENT_NAME).experiment_id
except:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

STUDY_DB = "sqlite:///local.study.db"


In [ ]:
def run_lgbm_bert(model_name, sim_col):
    """
    Оптимизирует LightGBM + ALS + BERT-similarity через Optuna,
    логирует в MLflow, возвращает dict с метриками.
    """
    short = model_name.split('/')[-1].replace('-','_').lower()
    features_bert = BASE_FEATURES + [sim_col, 'als_score']
    cat_bert = categorical_features  # те же категориальные

    X_tr = X_train[BASE_FEATURES + ['als_score']].copy()
    X_tr[sim_col] = df.loc[X_train.index, sim_col].values

    X_te = X_test[BASE_FEATURES + ['als_score']].copy()
    X_te[sim_col] = df.loc[X_test.index, sim_col].values

    def objective(trial):
        params = {
            'model__n_estimators':     trial.suggest_int('n_estimators', 100, 800, step=50),
            'model__max_depth':        trial.suggest_int('max_depth', 3, 12),
            'model__learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'model__num_leaves':       trial.suggest_int('num_leaves', 20, 200),
            'model__min_child_samples':trial.suggest_int('min_child_samples', 5, 80),
        }
        pipe = Pipeline([
            ('pre', ColumnTransformer([
                ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_bert)
            ], remainder='passthrough')),
            ('model', LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, class_weight='balanced'))
        ])
        pipe.set_params(**params)

        kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
        fold_ndcg = []
        for tr_idx, val_idx in kfold.split(X_tr, y_train):
            pipe.fit(X_tr.iloc[tr_idx], y_train.iloc[tr_idx])
            y_prob = pipe.predict_proba(X_tr.iloc[val_idx])[:, 1]
            df_val = df.loc[X_tr.iloc[val_idx].index].copy()
            df_val['y_pred_proba'] = y_prob
            m = calculate_metrics(df_val)
            fold_ndcg.append(m['NDCG'])
        return np.mean(fold_ndcg)

    study_name = f"LGBM_ALS_{short}"
    with mlflow.start_run(run_name=study_name, experiment_id=experiment_id) as parent:
        mlflc = MLflowCallback(
            tracking_uri=f'http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}',
            metric_name="NDCG", create_experiment=False,
            mlflow_kwargs={'experiment_id': experiment_id,
                           'tags': {MLFLOW_PARENT_RUN_ID: parent.info.run_id}})
        study = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
                                    study_name=study_name, storage=STUDY_DB,
                                    load_if_exists=True)
        study.optimize(objective, n_trials=15, callbacks=[mlflc])

    # Финальная модель с лучшими params
    best_pipe = Pipeline([
        ('pre', ColumnTransformer([
            ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_bert)
        ], remainder='passthrough')),
        ('model', LGBMClassifier(random_state=RANDOM_STATE, verbose=-1, class_weight='balanced'))
    ])
    best_pipe.set_params(**{f'model__{k}': v for k, v in study.best_params.items()})
    best_pipe.fit(X_tr, y_train)

    y_prob = best_pipe.predict_proba(X_te)[:, 1]
    df_te  = df.loc[X_te.index].copy()
    df_te['y_pred_proba'] = y_prob
    print(f"\n{model_name} — Test metrics:")
    metrics = calculate_metrics(df_te)

    with mlflow.start_run(run_name=f"best_{study_name}", experiment_id=experiment_id):
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('sim_col', sim_col)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(best_pipe, artifact_path='model')

    return {'Model': model_name, 'sim_col': sim_col,
            'Pipeline': best_pipe, **metrics}


In [ ]:
all_results = []

for model_name, _, _, _ in BERT_MODELS:
    sim_col = bert_sim_cols[model_name]
    print(f"\n{'='*60}\nЭксперимент: {model_name}\n{'='*60}")
    try:
        result = run_lgbm_bert(model_name, sim_col)
        all_results.append(result)
    except Exception as e:
        print(f"[ОШИБКА] {e}")
        all_results.append({'Model': model_name, 'sim_col': sim_col,
                             'NDCG': None, 'Pipeline': None})


## 9. Результаты

In [ ]:
NDCG_TFIDF_BASELINE = 0.7817  # LightGBM + ALS + TF-IDF из ML_Experiments.ipynb

results_df = pd.DataFrame([
    {'Model': r['Model'], 'NDCG': r['NDCG'],
     'Precision': r.get('Precision'), 'Recall': r.get('Recall'), 'F1': r.get('F1')}
    for r in all_results
])
results_df['Delta vs TF-IDF'] = results_df['NDCG'] - NDCG_TFIDF_BASELINE
results_df = results_df.sort_values('NDCG', ascending=False).reset_index(drop=True)

print(f"Базовая линия TF-IDF: NDCG = {NDCG_TFIDF_BASELINE}")
print()
print(results_df[['Model','NDCG','Delta vs TF-IDF']].to_string(index=False))


In [ ]:
valid = [r for r in all_results if r.get('NDCG') is not None]
if valid:
    best = max(valid, key=lambda r: r['NDCG'])
    if best['NDCG'] > NDCG_TFIDF_BASELINE:
        fname = f"pipeline_lgbm_als_{best['sim_col']}.pkl"
        with open(fname, 'wb') as f:
            pickle.dump(best['Pipeline'], f)
        print(f"Лучший пайплайн сохранён: {fname}")
        print(f"NDCG: {best['NDCG']:.4f}  (TF-IDF: {NDCG_TFIDF_BASELINE:.4f})")
    else:
        print(f"TF-IDF остаётся лучшим ({NDCG_TFIDF_BASELINE:.4f}).")
        print(f"Лучший BERT: {best['Model']}  NDCG={best['NDCG']:.4f}")
